# Tesseract OCR Benchmark

This notebook runs the full Tesseract benchmark on the Nepali Pixel dataset. It loads the dataset, processes images using the Tesseract adapter, and calculates metrics (CER, WER, Exact Match).

*(Note on GPU: As discussed, Tesseract inherently relies on CPU-based optimizations like OpenMP. If you require GPU acceleration, consider EasyOCR, Surya, or PaddleOCR models.)*

In [ ]:
import os
import sys
import time
import json
import statistics
import collections
import concurrent.futures
from datetime import datetime, timezone
from pathlib import Path

# Add the parent directory to sys.path so we can import the nepeval_ocr module
sys.path.append(os.path.abspath('.'))

from nepeval_ocr.dataset import load_nepali_pixel_dataset
from nepeval_ocr.scorers import compute_metrics
from nepeval_ocr.adapters.tesseract import TesseractAdapter

In [ ]:
# Configuration parameters for the benchmark
LANG = "nep"
CONCURRENCY = 4
LIMIT = None # Set to an integer (e.g., 50) for a quick smoke-test instead of the full dataset

def timestamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

output_dir = Path(f"results/tesseract_run_notebook_{timestamp()}")
model_name = f"tesseract-{LANG}"
model_dir = output_dir / "models" / model_name
model_dir.mkdir(parents=True, exist_ok=True)

samples_path = model_dir / "samples.jsonl"
progress_path = model_dir / "progress.json"

print(f"Results and progress will be saved to: {output_dir}")

In [ ]:
def evaluate_one(item, adapter):
    sample_id, image, ground_truth, metadata = item
    result = {
        "sample_id": sample_id,
        "ground_truth": ground_truth,
        "metadata": metadata,
        "model": f"tesseract-{adapter.lang}"
    }
    
    started = time.perf_counter()
    try:
        response_text = adapter.evaluate_sample(image)
        latency = time.perf_counter() - started
        metrics = compute_metrics(ground_truth, response_text)
        result.update({
            "ok": True,
            "latency_sec": latency,
            "response": response_text,
            "metrics": metrics,
        })
    except Exception as exc:
        result.update({
            "ok": False,
            "latency_sec": None,
            "response": "",
            "error": str(exc),
            "metrics": {"cer": 1.0, "wer": 1.0, "exact_match": False}
        })
    return result

def append_jsonl(handle, row) -> None:
    handle.write(json.dumps(row, ensure_ascii=False))
    handle.write("\n")
    handle.flush()

In [ ]:
# Load Dataset
print("Loading dataset...")
dataset_gen = load_nepali_pixel_dataset(split="train")
items = []
for i, item in enumerate(dataset_gen):
    if LIMIT and i >= LIMIT:
        break
    items.append(item)
    
print(f"Loaded {len(items)} items for benchmarking.")

In [ ]:
adapter = TesseractAdapter(lang=LANG)

# Pre-flight check
if items:
    print("Pre-flight: testing adapter on first sample...", flush=True)
    preflight = evaluate_one(items[0], adapter)
    if not preflight["ok"]:
        print(f"\n✗ Pre-flight check FAILED.\nError: {preflight['error']}\nAborting.")
        sys.exit(1)
    print("Pre-flight: OK", flush=True)

samples = []
started_at = datetime.now(timezone.utc).isoformat()

print(f"Starting benchmark with {CONCURRENCY} workers...")
with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENCY) as executor:
    futures = [executor.submit(evaluate_one, item, adapter) for item in items]
    with samples_path.open("w", encoding="utf-8") as samples_handle:
        for completed, future in enumerate(concurrent.futures.as_completed(futures), start=1):
            sample = future.result()
            samples.append(sample)
            append_jsonl(samples_handle, sample)
            
            if completed % 25 == 0 or completed == len(futures):
                progress = {
                    "model": model_name,
                    "completed_completions": completed,
                    "total_completions": len(futures),
                    "errored_completions": sum(1 for item in samples if not item["ok"]),
                    "updated_at": datetime.now(timezone.utc).isoformat(),
                }
                with progress_path.open("w") as f:
                    json.dump(progress, f)
                print(f"\rProgress: {completed}/{len(futures)} completions", end="", flush=True)

print("\nBenchmark finished!")

In [ ]:
# Summarize Results
valid_samples = [s for s in samples if s["ok"]]
errored_completions = len(samples) - len(valid_samples)

if samples and (errored_completions / len(samples)) > 0.5:
    error_counts = collections.Counter(s["error"] for s in samples if not s["ok"])
    print(f"\n✗ Run ABORTED: {errored_completions}/{len(samples)} samples errored.")
    for msg, count in error_counts.most_common():
        print(f"  [{count}x] {msg}")
else:
    summary = {
        "model": model_name,
        "started_at": started_at,
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "examples": len(samples),
        "errored_completions": errored_completions,
        "avg_cer": statistics.fmean(s["metrics"]["cer"] for s in valid_samples) if valid_samples else None,
        "avg_wer": statistics.fmean(s["metrics"]["wer"] for s in valid_samples) if valid_samples else None,
        "exact_match_rate": statistics.fmean(float(s["metrics"]["exact_match"]) for s in valid_samples) if valid_samples else None,
        "avg_latency_sec": statistics.fmean(s["latency_sec"] for s in valid_samples) if valid_samples else None,
    }

    with (model_dir / "summary.json").open("w") as f:
        json.dump(summary, f, indent=2)
        
    run_summary = {
        "run_id": output_dir.name,
        "models": [model_name],
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "model_summaries": [summary]
    }
    with (output_dir / "summary.json").open("w") as f:
        json.dump(run_summary, f, indent=2)

    print("\n=== Final Benchmark Summary ===")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")